# مساعد شركات الشحن (RAG + Chains + Output Parser) — مجاني 100% بموديل Hugging Face

بيجاوب على أي سؤال عن الشحن وشركات الشحن، ولو السؤال برا الموضوع بيعتذر بأدب.

**قبل التشغيل من Settings على اليمين:**
1. **Accelerator** ← **GPU T4 x2** (أو P100).
2. **Internet** ← On.

مفيش أي API key مطلوب.

## 1) تثبيت المكتبات

In [ ]:
!pip install -q langchain langchain-core langchain-community langchain-huggingface langchain-text-splitters sentence-transformers faiss-cpu pydantic gradio accelerate bitsandbytes

## 2) قاعدة المعرفة
كل مدخل = سؤال وجواب، وبينهم سطر فاضي. ضيف عليها بحرية (المعلومات دي أمثلة، والأفضل تبدّلها بمعلومات شركات حقيقية).

In [ ]:
from pathlib import Path

DATA_DIR = Path("/kaggle/working/data")
DATA_DIR.mkdir(exist_ok=True)

faq = """السؤال: ازاي أتتبع شحنتي؟
الجواب: بعد ما الشحنة بتتسلّم لشركة الشحن بيوصلك رقم تتبع على الإيميل أو SMS. دخّل الرقم في موقع شركة الشحن أو تطبيقها وهتشوف حالة الشحنة ومكانها.

السؤال: الشحنة اتأخرت، أعمل إيه؟
الجواب: راجع رقم التتبع الأول. لو الحالة واقفة أكتر من يومين تواصل مع شركة الشحن برقم الشحنة. التأخير ممكن يكون بسبب الجمارك أو الطقس أو عنوان ناقص أو ضغط المواسم.

السؤال: الشحنة وصلت تالفة، أعمل إيه؟
الجواب: صوّر الشحنة والتلف فوراً قبل ما تفتحها كاملة، وقدّم شكوى لشركة الشحن بسرعة (غالباً خلال أيام من الاستلام) ومعاك الصور والفاتورة. لو الشحنة كانت متأمّن عليها التعويض بيبقى أسهل.

السؤال: الشحنة ضاعت، أعمل إيه؟
الجواب: قدّم شكوى لشركة الشحن برقم الشحنة. الشركات عادة بتعمل تحقيق لمدة أيام، وبعدها لو ثبت الفقدان بيتم التعويض حسب سياستها أو قيمة التأمين.

السؤال: إيه الفرق بين الشحن البري والبحري والجوي؟
الجواب: الشحن البري مناسب للمسافات القريبة ومتوسط التكلفة. البحري الأرخص للكميات الكبيرة لكنه الأبطأ (أسابيع). الجوي الأسرع (أيام) والأغلى، ومناسب للبضائع الصغيرة أو المستعجلة.

السؤال: يعني إيه الوزن الحجمي؟
الجواب: شركات الشحن بتحسب السعر على الأكبر بين الوزن الفعلي والوزن الحجمي. الوزن الحجمي غالباً = (الطول × العرض × الارتفاع بالسم) ÷ 5000، والرقم بيختلف من شركة لأخرى.

السؤال: ازاي أحسب تكلفة الشحن؟
الجواب: التكلفة بتعتمد على الوزن (الفعلي أو الحجمي)، والمسافة أو الوجهة، ونوع الخدمة (عادي أو سريع)، وأي خدمات إضافية زي التأمين أو الدفع عند الاستلام. الأدق إنك تستخدم حاسبة الأسعار في موقع الشركة.

السؤال: يعني إيه الدفع عند الاستلام (COD)؟
الجواب: المستلم بيدفع ثمن البضاعة للمندوب وقت التسليم، والشركة بتحوّل المبلغ للتاجر بعد خصم رسومها. غالباً فيه رسوم إضافية على الخدمة دي.

السؤال: ايه الجمارك وازاي بتتحسب في الشحن الدولي؟
الجواب: الشحنات الدولية ممكن يتفرض عليها جمارك وضرائب حسب قيمة البضاعة ونوعها وبلد الوجهة. لازم فاتورة تجارية واضحة وبيانات صحيحة، وغالباً المستلم هو اللي بيدفعها لو ماتحسبتش على الشاحن.

السؤال: إيه المنتجات الممنوعة من الشحن؟
الجواب: غالباً ممنوع شحن المواد الخطرة والقابلة للاشتعال والمتفجرات والمخدرات والأسلحة وبعض البطاريات والسوائل. القائمة بتختلف من شركة لأخرى ومن بلد لبلد، فراجع شروط الشركة قبل الشحن.

السؤال: ازاي أغلّف الشحنة صح؟
الجواب: استخدم كرتون قوي ومواد حماية زي الفقاعات أو الفوم، واملا الفراغات، وقفّل الكرتون بشريط لاصق قوي. اكتب عنوان المستلم ورقم تليفونه بوضوح، وحط نسخة منه جوه الشحنة كمان.

السؤال: ينفع أغيّر عنوان الشحنة بعد ما اتشحنت؟
الجواب: ساعات ينفع لو الشحنة لسه في مرحلة مبكرة، بتواصل مع شركة الشحن برقم الشحنة. ممكن يتفرض رسوم إضافية أو يتأخر التسليم. بعد خروجها للتسليم غالباً بيبقى صعب.

السؤال: إيه مدة التوصيل المعتادة؟
الجواب: داخل نفس المدينة من يوم لـ 2 يوم عمل، وبين المحافظات من 2 لـ 5 أيام عمل، والشحن الدولي السريع من 3 لـ 7 أيام، والدولي العادي أو البحري من أسبوعين لأكتر. دي تقديرات والموعد الفعلي بيحدده مزوّد الخدمة.

السؤال: المستلم مكنش موجود وقت التوصيل، أعمل إيه؟
الجواب: المندوب غالباً بيحاول مرة أو اتنين تاني أو بيتواصل مع المستلم لتحديد معاد جديد. تقدر تتواصل مع الشركة تحدد معاد بديل أو تستلم من فرع الشركة لو متاح.

السؤال: ينفع أرجّع شحنة؟
الجواب: المرتجعات بتتم غالباً بالتنسيق بين التاجر وشركة الشحن، وفيه رسوم شحن للمرتجع حسب سياسة التاجر. المنتج لازم يرجع بحالته الأصلية وفي المدة المسموحة.

السؤال: إيه التأمين على الشحنة وهل أحتاجه؟
الجواب: التأمين بيعوّضك بقيمة البضاعة لو الشحنة ضاعت أو اتلفت، وغالباً بنسبة صغيرة من قيمتها. ينفع جداً للبضائع الغالية أو الحساسة، لأن تعويض الشركة الأساسي ممكن يكون محدود.

السؤال: إيه الشحن السريع (Express) وإيه الفرق عن العادي؟
الجواب: الشحن السريع بيدّي أولوية في المعالجة والتوصيل وبيوصل أسرع بكتير، وبتكلفة أعلى. العادي أرخص ومواعيده أطول وأقل مرونة.

السؤال: ازاي أشحن لبلد تانية؟
الجواب: اختار شركة شحن دولي، وجهّز فاتورة تجارية وبيانات المرسل والمستلم بدقة، وحدد نوع الخدمة، واتأكد من الجمارك وقائمة الممنوعات في بلد الوجهة. وبعد التسليم هتاخد رقم تتبع دولي.

السؤال: ايه الفرق بين شركات الشحن والمندوب الخاص؟
الجواب: شركات الشحن عندها شبكة وتتبع وتأمين وحدود مسؤولية واضحة، والمندوب الخاص غالباً أرخص للتوصيل المحلي لكن من غير تتبع رسمي أو ضمانات قوية.

السؤال: ساعات عمل خدمة العملاء؟
الجواب: (مثال) خدمة العملاء متاحة يومياً من 9 صباحاً حتى 9 مساءً ما عدا الجمعة.
"""

(DATA_DIR / "faq.txt").write_text(faq, encoding="utf-8")
print("تم إنشاء قاعدة المعرفة ✅")

## 3) تحميل الموديل + دالة تصنع LLM بحد أقصى للكلمات (للسرعة)
الافتراضي `Qwen2.5-3B-Instruct`. لو عايز جودة أعلى فعّل نسخة 7B المعلّقة تحت.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from langchain_core.runnables import RunnableLambda

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"   # ممكن تغيّره لموديلك بعد الـ Fine-tuning

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)

# --- خيار أقوى: 7B بضغط 4-bit (شيل الـ # من الأسطر دي وامسح الـ model اللي فوق) ---
# from transformers import BitsAndBytesConfig
# MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID, device_map="auto",
#     quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
# )


def make_llm(max_new_tokens: int):
    """بيرجّع LLM بيرد بالإجابة بس، وبحد أقصى لعدد الكلمات (كل ما قلّ كان أسرع)"""
    role = {"system": "system", "human": "user", "ai": "assistant"}

    def _generate(prompt_value) -> str:
        messages = [{"role": role.get(m.type, "user"), "content": m.content}
                    for m in prompt_value.to_messages()]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        new_tokens = out[0][inputs["input_ids"].shape[1]:]   # نقطع الـ prompt ونسيب الرد بس
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return RunnableLambda(_generate)


domain_llm = make_llm(5)     # yes/no بس
ticket_llm = make_llm(150)   # JSON قصير
answer_llm = make_llm(300)   # الإجابة
print("الموديل جاهز ✅")

## 4) الـ imports

In [ ]:
import re, json
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

## 5) شكل التذكرة + Chain التصنيف (Output Parser)

In [ ]:
class Ticket(BaseModel):
    category: Literal["shipping", "payment", "returns", "account", "technical", "other"] = Field(
        description="نوع المشكلة"
    )
    priority: Literal["low", "medium", "high"] = Field(description="أولوية التذكرة")
    sentiment: Literal["positive", "neutral", "negative"] = Field(description="مشاعر العميل")
    summary: str = Field(description="ملخص المشكلة في جملة واحدة")
    needs_human: bool = Field(description="هل تحتاج تدخل موظف بشري؟")


ticket_parser = PydanticOutputParser(pydantic_object=Ticket)

ticket_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "أنت نظام فرز تذاكر دعم فني. حلل رسالة العميل وارجع JSON فقط بدون أي شرح إضافي.\n"
     "{format_instructions}"),
    ("human", "{message}"),
]).partial(format_instructions=ticket_parser.get_format_instructions())

ticket_chain = ticket_prompt | ticket_llm | StrOutputParser()


def classify(message: str) -> Ticket:
    raw = ticket_chain.invoke({"message": message})
    match = re.search(r"\{.*\}", raw, re.DOTALL)   # نطلّع الـ JSON من الرد
    try:
        return ticket_parser.parse(match.group(0) if match else raw)
    except Exception:
        return Ticket(category="other", priority="medium", sentiment="neutral",
                      summary=message[:100], needs_human=True)

## 6) RAG: كل سؤال وجواب = قطعة مستقلة

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # بيدعم العربي
)

raw = (DATA_DIR / "faq.txt").read_text(encoding="utf-8")
chunks = [Document(page_content=b.strip()) for b in raw.split("\n\n") if b.strip()]

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"عدد المداخل: {len(chunks)} ✅")

## 7) فلتر الموضوع + Chain الإجابة
أي سؤال له علاقة بالشحن بيتجاوب (من القاعدة أو من المعرفة العامة)، واللي برا الموضوع بيتحوّل بأدب.

In [ ]:
domain_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "حدد هل رسالة العميل ليها علاقة بالشحن أو التوصيل أو شركات الشحن أو تتبع الشحنات أو "
     "الجمارك أو التغليف أو الدفع عند الاستلام أو الاسترجاع.\n"
     "جاوب بكلمة واحدة فقط: yes أو no. جاوب no فقط لو الرسالة واضح تماماً إنها بعيدة عن الموضوع، "
     "ولو فيه أي شك جاوب yes.\n"
     "أمثلة:\n"
     "'شحنتي اتأخرت' -> yes\n"
     "'ايه الفرق بين الشحن البحري والجوي' -> yes\n"
     "'كتبت العنوان غلط اعمل ايه' -> yes\n"
     "'المندوب مجاش' -> yes\n"
     "'ازاي اعمل كبسة' -> no\n"
     "'مين فاز في الماتش' -> no"),
    ("human", "{message}"),
])
domain_chain = domain_prompt | domain_llm | StrOutputParser()


SHIPPING_KEYWORDS = [
    "شحن", "توصيل", "مندوب", "تتبع", "طلب", "عنوان", "جمارك", "تغليف", "استلام", "تسليم",
    "ارجاع", "إرجاع", "استرجاع", "كارتون", "طرد", "بوسطة", "ارامكس", "أرامكس", "دي اتش", "فيديكس",
    "ship", "deliver", "package", "parcel", "track", "courier", "customs", "address", "order",
]


def is_in_domain(message: str) -> bool:
    msg = message.lower()
    if any(k in msg for k in SHIPPING_KEYWORDS):      # كلمة واضحة من عالم الشحن -> يجاوب علطول
        return True
    out = domain_chain.invoke({"message": message}).strip().lower()
    return not out.startswith("no")                    # لو الموديل مش متأكد، نفترض إنه في الموضوع


answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "أنت مساعد خبير في الشحن وشركات الشحن وخدمة العملاء. جاوب بنفس لغة العميل وبشكل مختصر ومهذب.\n"
     "- لو المعلومة موجودة في السياق، اعتمد عليها أولاً.\n"
     "- لو السؤال عام عن الشحن وملقيتش الإجابة في السياق، جاوب من معرفتك العامة.\n"
     "- لو السؤال عن أسعار أو مواعيد أو أرقام خاصة بشركة بعينها ومش في السياق، قول إنك مش متأكد "
     "وانصحه يراجع الموقع الرسمي للشركة. ممنوع تخترع أرقام.\n\nالسياق:\n{context}"),
    ("human", "{message}"),
])
answer_chain = answer_prompt | answer_llm | StrOutputParser()


def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

## 8) الدالة الرئيسية

In [ ]:
def handle_message(message: str) -> dict:
    if not is_in_domain(message):
        return {
            "answer": "أنا متخصص في أسئلة الشحن وشركات الشحن. اسألني فيها وأنا في الخدمة 🙂",
            "ticket": None,
            "sources": [],
        }

    ticket = classify(message)
    docs = retriever.invoke(message)
    answer = answer_chain.invoke({"message": message, "context": format_docs(docs)})
    return {
        "answer": answer.strip(),
        "ticket": ticket.model_dump(),
        "sources": [d.page_content[:200] for d in docs],
    }

## 9) تجربة

In [ ]:
tests = [
    "ايه الفرق بين الشحن البحري والجوي؟",
    "طلبي اتأخر وخصموا الفلوس من الفيزا وأنا زهقت!",
    "ازاي اعمل كبسة؟",
]
for q in tests:
    r = handle_message(q)
    print("السؤال:", q)
    print("الرد:", r["answer"])
    print("التذكرة:", json.dumps(r["ticket"], ensure_ascii=False))
    print("-" * 40)

## 10) واجهة تفاعلية (Gradio)

In [ ]:
import gradio as gr

def chat(message):
    r = handle_message(message)
    ticket = json.dumps(r["ticket"], ensure_ascii=False, indent=2) if r["ticket"] else "—"
    sources = "\n\n".join(f"📄 {s}..." for s in r["sources"]) or "—"
    return r["answer"], ticket, sources

demo = gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label="رسالة العميل", lines=3),
    outputs=[gr.Textbox(label="الإجابة"), gr.Code(label="التذكرة", language="json"), gr.Textbox(label="المصادر")],
    title="🚚 مساعد شركات الشحن",
)
demo.launch(share=True)